<a href="https://colab.research.google.com/github/ZainAliShah199/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainAliShah199/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**My baseline rule identifies pages that should be reviewed for content refresh.**

**The rule gives a higher score to pages that:**
- have not been updated for at least 180 days,
- receive a good number of impressions,
- have a low CTR,
- rank beyond the first few search positions.

**The action for the highest scoring pages is "Review for Content Refresh".**

**Reason Codes:**
- **STALE_CONTENT** – Page has not been updated recently.
- **LOW_CTR** – CTR is lower than expected.
- **HIGH_VISIBILITY** – Page receives many impressions.
- **LOW_RANK** – Average search position is relatively poor.

**Action Label:**
REVIEW_CONTENT

In [15]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

# Clone repo if not already available
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Move into repo
os.chdir(REPO_DIR)

# Install requirements
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True
)

print("Working directory:", os.getcwd())

# Check dataset exists
assert os.path.exists(
    "data/raw/content_refresh_anonymized.csv"
), "Dataset not found"

print("Dataset found successfully!")

Working directory: /flyrank-ml-internship-starter/flyrank-ml-internship-starter
Dataset found successfully!


In [16]:
import os
import pandas as pd

# Locate project root
while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


**This baseline score combines four observable signals.**

**The score increases when:**
- the content is stale,
- impressions are high,
- CTR is low,
- average position is poor.

The queue is sorted from highest score to lowest score.

**Each page receives:**
- Baseline Score
- Reason Code
- Action Label

The notebook writes the ranked queue to:

work/outputs/baseline_action_score.csv

In [17]:
import os

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Baseline score
df["baseline_score"] = (
    (df["days_since_last_update"] >= 180).astype(int) * 40 +
    (df["impressions_90d"] >= 500).astype(int) * 30 +
    (df["ctr"] < 0.05).astype(int) * 20 +
    (df["avg_position"] > 10).astype(int) * 10
)

# Reason Code
def reason(row):
    reasons = []

    if row["days_since_last_update"] >= 180:
        reasons.append("STALE_CONTENT")

    if row["ctr"] < 0.05:
        reasons.append("LOW_CTR")

    if row["impressions_90d"] >= 500:
        reasons.append("HIGH_VISIBILITY")

    if row["avg_position"] > 10:
        reasons.append("LOW_RANK")

    return ", ".join(reasons)

df["reason_code"] = df.apply(reason, axis=1)

# Action Label
df["action_label"] = "REVIEW_CONTENT"

queue = df.sort_values("baseline_score", ascending=False)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully.")
print(queue[[
    "baseline_score",
    "reason_code",
    "action_label"
]].head())

CSV saved successfully.
       baseline_score                                        reason_code  \
11489             100  STALE_CONTENT, LOW_CTR, HIGH_VISIBILITY, LOW_RANK   
3507              100  STALE_CONTENT, LOW_CTR, HIGH_VISIBILITY, LOW_RANK   
698               100  STALE_CONTENT, LOW_CTR, HIGH_VISIBILITY, LOW_RANK   
11630              80           STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK   
5327               80           STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK   

         action_label  
11489  REVIEW_CONTENT  
3507   REVIEW_CONTENT  
698    REVIEW_CONTENT  
11630  REVIEW_CONTENT  
5327   REVIEW_CONTENT  


**The Top-20 pages all receive the action label REVIEW_CONTENT because they have the highest baseline scores.**

Reason codes identify why each page was selected.

Confidence is Medium because this is a rule-based baseline rather than a machine learning model.

**The recommendation could be wrong if:**
- seasonal traffic affected impressions,
- the page was recently improved but performance has not updated yet,
- the content serves a stable niche where low CTR is expected,
- user intent has changed.

In [18]:
top20 = queue.head(20).copy()

top20["confidence"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "Seasonality, recent updates, niche content, or temporary ranking changes."
)

review = top20[[
    "baseline_score",
    "reason_code",
    "action_label",
    "confidence",
    "what_would_make_it_wrong"
]]

print(review)

review

       baseline_score                                        reason_code  \
11489             100  STALE_CONTENT, LOW_CTR, HIGH_VISIBILITY, LOW_RANK   
3507              100  STALE_CONTENT, LOW_CTR, HIGH_VISIBILITY, LOW_RANK   
698               100  STALE_CONTENT, LOW_CTR, HIGH_VISIBILITY, LOW_RANK   
11630              80           STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK   
5327               80           STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK   
7021               80           STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK   
23215              80           STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK   
20837              80           STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK   
26799              80           STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK   
16751              80           STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK   
21268              80           STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK   
16514              80           STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK   
12045       

,baseline_score,reason_code,action_label,confidence,what_would_make_it_wrong
11489,100,"STALE_CONTENT, LOW_CTR, HIGH_VISIBILITY, LOW_RANK",REVIEW_CONTENT,Medium,"Seasonality, recent updates, niche content, or..."
3507,100,"STALE_CONTENT, LOW_CTR, HIGH_VISIBILITY, LOW_RANK",REVIEW_CONTENT,Medium,"Seasonality, recent updates, niche content, or..."
698,100,"STALE_CONTENT, LOW_CTR, HIGH_VISIBILITY, LOW_RANK",REVIEW_CONTENT,Medium,"Seasonality, recent updates, niche content, or..."
11630,80,"STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK",REVIEW_CONTENT,Medium,"Seasonality, recent updates, niche content, or..."
5327,80,"STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK",REVIEW_CONTENT,Medium,"Seasonality, recent updates, niche content, or..."
7021,80,"STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK",REVIEW_CONTENT,Medium,"Seasonality, recent updates, niche content, or..."
23215,80,"STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK",REVIEW_CONTENT,Medium,"Seasonality, recent updates, niche content, or..."
20837,80,"STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK",REVIEW_CONTENT,Medium,"Seasonality, recent updates, niche content, or..."
26799,80,"STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK",REVIEW_CONTENT,Medium,"Seasonality, recent updates, niche content, or..."
16751,80,"STALE_CONTENT, HIGH_VISIBILITY, LOW_RANK",REVIEW_CONTENT,Medium,"Seasonality, recent updates, niche content, or..."


**Some pages may receive a high baseline score even though they do not truly require content refresh.**

**Possible weak picks include:**
- seasonal pages,
- pages with temporary traffic fluctuations,
- pages that already have planned updates.

No product flags or future information were used.

**The baseline uses only observable features available before making the decision:**
- impressions,
- CTR,
- average position,
- days since last update.

No label-derived columns such as trend_direction or trend_pct were used, so leakage was avoided.

In [19]:
print("Leakage Check")

used_features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

print("Features used:")
print(used_features)

print("\nNo future-window variables used.")
print("No product flags used.")
print("No trend_direction used.")
print("No trend_pct used.")
print("\nLeakage Check: PASSED")

Leakage Check
Features used:
['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position']

No future-window variables used.
No product flags used.
No trend_direction used.
No trend_pct used.

Leakage Check: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.